## Flood Prediction

This project predicts the probability of flooding in a region from 20 factors related to floods (monsoon intensity, deforestation, urbanization, drainage systems, dams, population score, and more). It is a regression problem scored by R² (coefficient of determination).

## Approach
1. Load the data and explore the target and features
2. Train a simple linear regression baseline and evaluate R² on a validation split
3. Generate predictions and create the submission file
4. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c playground-series-s4e5
!unzip -oq playground-series-s4e5.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 7.6 MB/s eta 0:00:00
100% 28.0M/28.0M [00:00<00:00, 91.0MB/s]



In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
train.shape, test.shape

((1117957, 22), (745305, 21))

In [4]:
train.sample(1)

,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
60454,60454,4,1,3,6,5,5,8,7,6,...,7,4,5,6,3,5,3,8,2,0.52


In [5]:
test.sample(1)

,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,IneffectiveDisasterPreparedness,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors
193032,1310989,4,5,4,7,7,3,5,7,3,...,6,9,6,3,6,4,4,1,4,5


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1117957 entries, 0 to 1117956
Data columns (total 22 columns):
 #   Column                           Non-Null Count    Dtype  
---  ------                           --------------    -----  
 0   id                               1117957 non-null  int64  
 1   MonsoonIntensity                 1117957 non-null  int64  
 2   TopographyDrainage               1117957 non-null  int64  
 3   RiverManagement                  1117957 non-null  int64  
 4   Deforestation                    1117957 non-null  int64  
 5   Urbanization                     1117957 non-null  int64  
 6   ClimateChange                    1117957 non-null  int64  
 7   DamsQuality                      1117957 non-null  int64  
 8   Siltation                        1117957 non-null  int64  
 9   AgriculturalPractices            1117957 non-null  int64  
 10  Encroachments                    1117957 non-null  int64  
 11  IneffectiveDisasterPreparedness  1117957 non-null 

In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 745305 entries, 0 to 745304
Data columns (total 21 columns):
 #   Column                           Non-Null Count   Dtype
---  ------                           --------------   -----
 0   id                               745305 non-null  int64
 1   MonsoonIntensity                 745305 non-null  int64
 2   TopographyDrainage               745305 non-null  int64
 3   RiverManagement                  745305 non-null  int64
 4   Deforestation                    745305 non-null  int64
 5   Urbanization                     745305 non-null  int64
 6   ClimateChange                    745305 non-null  int64
 7   DamsQuality                      745305 non-null  int64
 8   Siltation                        745305 non-null  int64
 9   AgriculturalPractices            745305 non-null  int64
 10  Encroachments                    745305 non-null  int64
 11  IneffectiveDisasterPreparedness  745305 non-null  int64
 12  DrainageSystems               

In [8]:
train.columns.tolist()

['id',
 'MonsoonIntensity',
 'TopographyDrainage',
 'RiverManagement',
 'Deforestation',
 'Urbanization',
 'ClimateChange',
 'DamsQuality',
 'Siltation',
 'AgriculturalPractices',
 'Encroachments',
 'IneffectiveDisasterPreparedness',
 'DrainageSystems',
 'CoastalVulnerability',
 'Landslides',
 'Watersheds',
 'DeterioratingInfrastructure',
 'PopulationScore',
 'WetlandLoss',
 'InadequatePlanning',
 'PoliticalFactors',
 'FloodProbability']

In [9]:
train["FloodProbability"].describe()

,FloodProbability
count,1.117957e+06
mean,5.044803e-01
std,5.102610e-02
min,2.850000e-01
25%,4.700000e-01
50%,5.050000e-01
75%,5.400000e-01
max,7.250000e-01


In [10]:
test_ids = test["id"]

x = train.drop(columns=["id", "FloodProbability"])
y = train["FloodProbability"]

In [11]:
x.shape, y.shape

((1117957, 20), (1117957,))

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(x_train, y_train)

pred = model.predict(x_val)
print("Validation R2:", round(r2_score(y_val, pred), 4))

Validation R2: 0.8449


In [13]:
model.fit(x, y)

x_test = test.drop(columns=["id"])
pred_test = model.predict(x_test)

pd.DataFrame({"id": test_ids, "FloodProbability": pred_test}).to_csv("submission.csv", index=False)

In [14]:
!kaggle competitions submit -c playground-series-s4e5 -f submission.csv -m "LinearRegression baseline"

100% 19.3M/19.3M [00:00<00:00, 36.0MB/s]
99 submissions remaining today.
Successfully submitted to Regression with a Flood Prediction Dataset

In [15]:
import pickle, os
pickle.dump(model, open("flood_model.pkl", "wb"))
print("size KB:", os.path.getsize("flood_model.pkl") // 1024)

size KB: 1
